# Causal Inference in Practice
## Week 3 — Randomized Experiments & A/B Testing · Practice Notebook

> **Block I — Foundations**
>
> Designing and analyzing experiments — the cleanest route to causation — and the failure modes that appear at scale.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · Random assignment makes the difference in means unbiased

We simulate an experiment where we **know** the true average treatment effect is `2.0`. A pre-period covariate `pre` drives the baseline outcome (we'll exploit it later for CUPED). Because treatment `Z` is assigned by a coin flip, `Z` is independent of the potential outcomes — so the plain difference in means `E[Y|Z=1] − E[Y|Z=0]` is an unbiased estimate of the effect.

In [ ]:
import statsmodels.api as sm
from scipy import stats

TRUE_ATE = 2.0

def simulate_experiment(n=8000, ate=TRUE_ATE):
    """One randomized experiment with a known additive effect."""
    pre  = RNG.normal(50, 10, n)                 # pre-period covariate
    base = 0.6 * pre + RNG.normal(0, 8, n)       # baseline outcome
    Z    = RNG.binomial(1, 0.5, n)               # randomized 50/50
    Y    = base + ate * Z + RNG.normal(0, 5, n)  # TRUE effect = ate
    return pd.DataFrame({'Z': Z, 'Y': Y, 'pre': pre})

df = simulate_experiment()
yt = df.loc[df.Z == 1, 'Y']
yc = df.loc[df.Z == 0, 'Y']
ate = yt.mean() - yc.mean()
print(f'difference in means = {ate:.3f}   (truth = {TRUE_ATE})')

A single experiment is noisy. The real claim is that the estimator is **unbiased** — averaged over many experiments it lands on the truth. Let's run 400 fresh experiments and check the mean of the estimates.

In [ ]:
ests = []
for _ in range(400):
    d = simulate_experiment()
    ests.append(d.loc[d.Z == 1, 'Y'].mean() - d.loc[d.Z == 0, 'Y'].mean())
ests = np.array(ests)
print(f'mean of 400 estimates = {ests.mean():.3f}   (truth = {TRUE_ATE})')
print(f'std of the estimates  = {ests.std():.3f}')
assert abs(ests.mean() - TRUE_ATE) < 0.1, 'estimator should be unbiased'

### Build the confidence interval

The standard error of a difference in means is `sqrt(var_t/n_t + var_c/n_c)`. The 95% CI is `ate ± 1.96·SE`. On a single experiment we expect the truth to fall inside ~95% of such intervals.

In [ ]:
df = simulate_experiment()
yt = df.loc[df.Z == 1, 'Y']
yc = df.loc[df.Z == 0, 'Y']
ate = yt.mean() - yc.mean()
se  = np.sqrt(yt.var(ddof=1)/yt.size + yc.var(ddof=1)/yc.size)
lo, hi = ate - 1.96*se, ate + 1.96*se
print(f'ATE = {ate:.3f}   SE = {se:.3f}   95% CI = [{lo:.3f}, {hi:.3f}]')
print(f'truth {TRUE_ATE} inside CI? {lo <= TRUE_ATE <= hi}')

### 🔧 Exercise 1.1 — recover the same estimate with OLS

Regressing `Y` on a constant and `Z` gives the difference in means as the coefficient on `Z`, plus a standard error and CI for free. Fit `sm.OLS` and pull out the `Z` coefficient. Confirm it matches the hand-computed `ate` above.

Fill in the `# TODO`s below.

In [ ]:
# TODO: regress Y on a constant and Z, then read off the slope on Z.
X = sm.add_constant(df['Z'].to_numpy(dtype=float))   # design matrix
beta_Z = ...        # TODO: fit sm.OLS(df['Y'], X) and take params[1]
# print(f'OLS coefficient on Z = {beta_Z:.3f}')

### ✅ Solution 1.1

In [ ]:
X = sm.add_constant(df['Z'].to_numpy(dtype=float))
fit = sm.OLS(df['Y'].to_numpy(dtype=float), X).fit()
beta_Z = fit.params[1]
print(f'OLS coefficient on Z = {beta_Z:.3f}   (hand-computed {ate:.3f})')
assert abs(beta_Z - ate) < 1e-6, 'OLS slope on Z IS the difference in means'

## 2 · Power and the minimum detectable effect (MDE)

Before launching, you size the test. For a two-proportion test (conversion at baseline `p0` vs `p1 = p0 + MDE`), the per-arm sample size has a closed form. We compute it, then **verify it by simulation**: at that `n`, the empirical power should be ≈ the target we asked for.

In [ ]:
p0, mde, alpha, power = 0.10, 0.02, 0.05, 0.80
p1 = p0 + mde
za = stats.norm.ppf(1 - alpha/2)      # 1.96 for a two-sided 5% test
zb = stats.norm.ppf(power)            # 0.84 for 80% power
pbar = (p0 + p1) / 2
n_per_arm = int(np.ceil(
    (za*np.sqrt(2*pbar*(1-pbar)) + zb*np.sqrt(p0*(1-p0)+p1*(1-p1)))**2
    / mde**2))
print(f'baseline {p0:.0%}, MDE {mde:.0%}, power {power:.0%}')
print(f'required n per arm = {n_per_arm:,}')

Now simulate: draw two arms of size `n_per_arm` with the **true** rates `p0` and `p1`, run a two-proportion z-test, and repeat. The fraction of trials that reject the null is the empirical power — it should land near `0.80`.

In [ ]:
def trial_rejects(n, p0, p1):
    a = RNG.binomial(1, p0, n)
    b = RNG.binomial(1, p1, n)
    pa, pb = a.mean(), b.mean()
    pp = (a.sum() + b.sum()) / (2*n)             # pooled rate
    se = np.sqrt(pp*(1-pp) * (2/n))
    return se > 0 and abs((pb - pa) / se) > za   # reject H0?

reps = 2000
emp_power = np.mean([trial_rejects(n_per_arm, p0, p1) for _ in range(reps)])
print(f'empirical power at n={n_per_arm:,}: {emp_power:.3f}  (target {power})')
assert abs(emp_power - power) < 0.06, 'sizing formula should hit the target power'

### 🔧 Exercise 2.1 — halving the MDE

Detectable effects shrink like `1/√n`, so the sample size scales like `1/MDE²`. Recompute `n_per_arm` for a **1-point** lift (`mde = 0.01`) and confirm it is roughly **four times** the 2-point sample.

Fill in the `# TODO`s below.

In [ ]:
def sample_size(p0, mde, alpha=0.05, power=0.80):
    p1 = p0 + mde
    za = stats.norm.ppf(1 - alpha/2)
    zb = stats.norm.ppf(power)
    pbar = (p0 + p1) / 2
    return int(np.ceil(
        (za*np.sqrt(2*pbar*(1-pbar)) + zb*np.sqrt(p0*(1-p0)+p1*(1-p1)))**2
        / mde**2))

n_2pt = sample_size(0.10, 0.02)
n_1pt = ...        # TODO: call sample_size with mde = 0.01
# print(n_2pt, n_1pt, n_1pt / n_2pt)

### ✅ Solution 2.1

In [ ]:
n_2pt = sample_size(0.10, 0.02)
n_1pt = sample_size(0.10, 0.01)
print(f'n per arm: 2-pt lift = {n_2pt:,}   1-pt lift = {n_1pt:,}')
print(f'ratio = {n_1pt / n_2pt:.2f}x  (≈ 4 — quartering the MDE squares the cost)')
assert 3.5 < n_1pt / n_2pt < 4.5, 'halving the MDE ~quadruples n'

## 3 · CUPED — variance reduction from a pre-period covariate

We have a covariate `pre` measured **before** treatment. CUPED subtracts the predictable part of `Y`:

`Y_cuped = Y − θ·(pre − mean(pre))`, with `θ = Cov(Y, pre)/Var(pre)`.

Because `pre` is pre-treatment, it is balanced across arms, so the **estimate stays unbiased** — but the residual outcome has less variance, so the **standard error shrinks**. We assert both.

In [ ]:
df = simulate_experiment(n=8000)

# --- naive difference in means ---
yt, yc = df.loc[df.Z==1, 'Y'], df.loc[df.Z==0, 'Y']
ate_naive = yt.mean() - yc.mean()
se_naive  = np.sqrt(yt.var(ddof=1)/yt.size + yc.var(ddof=1)/yc.size)

# --- CUPED-adjusted outcome ---
theta = np.cov(df['Y'], df['pre'])[0, 1] / df['pre'].var(ddof=1)
df = df.assign(Ycuped=df['Y'] - theta*(df['pre'] - df['pre'].mean()))
at, ac = df.loc[df.Z==1, 'Ycuped'], df.loc[df.Z==0, 'Ycuped']
ate_cuped = at.mean() - ac.mean()
se_cuped  = np.sqrt(at.var(ddof=1)/at.size + ac.var(ddof=1)/ac.size)

print(f'naive : ATE = {ate_naive:.3f}   SE = {se_naive:.4f}')
print(f'CUPED : ATE = {ate_cuped:.3f}   SE = {se_cuped:.4f}')
print(f'SE reduction = {100*(1 - se_cuped/se_naive):.1f}%   (truth {TRUE_ATE})')

assert abs(ate_cuped - TRUE_ATE) < 0.6, 'CUPED estimate stays unbiased'
assert se_cuped < se_naive, 'CUPED must reduce the standard error'

The CUPED estimate is still ~`2.0` but its SE is meaningfully smaller — that is *free power*. The stronger the pre-period covariate correlates with the outcome, the bigger the win. Let's visualize the two sampling distributions to make the variance drop concrete.

In [ ]:
naive_draws, cuped_draws = [], []
for _ in range(300):
    d = simulate_experiment(n=4000)
    th = np.cov(d['Y'], d['pre'])[0, 1] / d['pre'].var(ddof=1)
    yc_ = d['Y'] - th*(d['pre'] - d['pre'].mean())
    naive_draws.append(d.loc[d.Z==1,'Y'].mean() - d.loc[d.Z==0,'Y'].mean())
    cuped_draws.append(yc_[d.Z==1].mean() - yc_[d.Z==0].mean())

fig, ax = plt.subplots()
ax.hist(naive_draws, bins=30, alpha=0.5, label='naive')
ax.hist(cuped_draws, bins=30, alpha=0.5, label='CUPED')
ax.axvline(TRUE_ATE, color='k', ls='--', label='truth')
ax.set_title('CUPED narrows the sampling distribution (same center)')
ax.legend()
print(f'std(naive) = {np.std(naive_draws):.3f}   '
      f'std(CUPED) = {np.std(cuped_draws):.3f}')
assert np.std(cuped_draws) < np.std(naive_draws)

### 🔧 Exercise 3.1 — covariate-adjusted OLS gives the same gain

Instead of transforming the outcome, regress `Y` on `Z` **and** the pre-period covariate `pre`. The coefficient on `Z` is the adjusted ATE, and its reported standard error should be close to the CUPED SE — same idea, different bookkeeping.

Fill in the `# TODO`s below.

In [ ]:
# TODO: build the design matrix [const, Z, pre] and fit OLS.
Xmat = sm.add_constant(np.column_stack([df['Z'], df['pre']]))
fit_adj = ...      # TODO: sm.OLS(df['Y'], Xmat).fit()
# beta_Z   = fit_adj.params[1]
# se_Z     = fit_adj.bse[1]
# print(beta_Z, se_Z)

### ✅ Solution 3.1

In [ ]:
Xmat = sm.add_constant(np.column_stack([df['Z'], df['pre']]))
fit_adj = sm.OLS(df['Y'].to_numpy(dtype=float), Xmat).fit()
beta_Z = fit_adj.params[1]
se_Z   = fit_adj.bse[1]
print(f'covariate-adjusted ATE = {beta_Z:.3f}   SE = {se_Z:.4f}')
print(f'CUPED SE was {se_cuped:.4f} — same ballpark, both << naive {se_naive:.4f}')
assert abs(beta_Z - TRUE_ATE) < 0.6, 'adjusted estimate stays unbiased'
assert se_Z < se_naive, 'covariate adjustment also reduces the SE'

## 4 · Peeking inflates the false-positive rate

Now a **true null**: treatment and control have the **same** rate, so any 'significant' result is a false positive. We compare two analysts. One looks **once** at the end. The other **peeks** at the running result many times and stops the instant it crosses ±1.96. We measure how often each falsely rejects.

In [ ]:
def run_with_peeks(n_final, p, n_peeks):
    """Sequential experiment under a TRUE NULL; reject at first 'sig' look."""
    a = RNG.binomial(1, p, n_final)
    b = RNG.binomial(1, p, n_final)
    ca, cb = np.cumsum(a), np.cumsum(b)
    look_at = np.linspace(n_final // n_peeks, n_final, n_peeks).astype(int)
    for t in look_at:
        pa, pb = ca[t-1]/t, cb[t-1]/t
        pp = (ca[t-1] + cb[t-1]) / (2*t)
        se = np.sqrt(pp*(1-pp) * (2/t))
        if se > 0 and abs((pb - pa) / se) > 1.96:
            return True       # falsely declared a winner
    return False

reps = 1500
fpr_once  = np.mean([run_with_peeks(2000, 0.10, 1)  for _ in range(reps)])
fpr_peek  = np.mean([run_with_peeks(2000, 0.10, 10) for _ in range(reps)])
print(f'false-positive rate, look ONCE   : {fpr_once:.3f}  (should be ~0.05)')
print(f'false-positive rate, 10 PEEKS    : {fpr_peek:.3f}  (inflated!)')
assert fpr_once < 0.09, 'a single fixed-n look controls alpha near 5%'
assert fpr_peek > fpr_once + 0.05, 'peeking must inflate the false-positive rate'

Looking once controls the error rate near the nominal 5%. Peeking ten times pushes it far higher — every extra look is another chance for the wandering statistic to cross the line under a true null. The fix is to fix `n` in advance and look once, or to use a sequential testing method designed for continuous monitoring.

### 🔧 Exercise 4.1 — error rate grows with the number of looks

Sweep the number of peeks over `[1, 5, 10, 20]` and record the false-positive rate for each. Confirm it is **monotonically increasing** in the number of looks.

Fill in the `# TODO`s below.

In [ ]:
peek_counts = [1, 5, 10, 20]
fprs = []
for k in peek_counts:
    # TODO: estimate the FPR with k peeks over `reps` true-null experiments
    rate = ...   # TODO: np.mean([run_with_peeks(2000, 0.10, k) for _ in range(reps)])
    fprs.append(rate)
# print(list(zip(peek_counts, fprs)))

### ✅ Solution 4.1

In [ ]:
peek_counts = [1, 5, 10, 20]
fprs = [np.mean([run_with_peeks(2000, 0.10, k) for _ in range(reps)])
        for k in peek_counts]
for k, r in zip(peek_counts, fprs):
    print(f'{k:2d} peeks -> false-positive rate {r:.3f}')

fig, ax = plt.subplots()
ax.plot(peek_counts, fprs, marker='o')
ax.axhline(0.05, color='k', ls='--', label='nominal 5%')
ax.set_xlabel('number of looks'); ax.set_ylabel('false-positive rate')
ax.set_title('More peeking → more false positives'); ax.legend()
assert fprs[-1] > fprs[0], 'FPR should rise with the number of looks'

## 5 · ITT vs. per-protocol under noncompliance

Finally, a leaky experiment. Treatment is randomly **assigned** (`Z`), but only some assigned units actually **comply** and receive it (`D`). Here healthier units are more likely to comply, and health also boosts the outcome — so compliance is confounded with the outcome. The true effect on those who take the treatment is `3.0`.

In [ ]:
n = 30000; TRUE = 3.0
Z = RNG.binomial(1, 0.5, n)                     # randomized assignment
health = RNG.normal(size=n)                     # drives compliance AND outcome
p_comply = 1 / (1 + np.exp(-(0.5 + 1.0*health)))
comply = RNG.binomial(1, p_comply, n)
D = Z * comply                                  # treatment RECEIVED (one-sided)
Y = 10 + 4*health + TRUE*D + RNG.normal(0, 3, n)

itt = Y[Z==1].mean() - Y[Z==0].mean()           # by ASSIGNMENT
pp  = Y[D==1].mean() - Y[D==0].mean()           # by treatment RECEIVED
compliance = D[Z==1].mean()                     # share who comply when assigned
cace = itt / compliance                         # IV / effect on compliers

print(f'compliance rate          = {compliance:.3f}')
print(f'ITT (by assignment)      = {itt:.3f}   (unbiased for the OFFER, diluted)')
print(f'per-protocol (by uptake) = {pp:.3f}   (BIASED — confounded by health)')
print(f'CACE = ITT / compliance  = {cace:.3f}   (truth on compliers = {TRUE})')

assert abs(cace - TRUE) < 0.5, 'ITT scaled by compliance recovers the compliers effect'
assert pp > TRUE + 0.8, 'per-protocol is inflated by healthy compliers'

Three numbers from one dataset. **ITT** is below the true `3.0` because 40% of the assigned never took the treatment — but it is the honest, unbiased effect of *offering* the program. **Per-protocol** is inflated because compliers are healthier than non-compliers — randomization no longer protects that comparison. **CACE** (`ITT / compliance`) rescales the diluted ITT and recovers the effect on compliers. We formalize this instrument in Week 9.

## Wrap-up & self-check

- **Random assignment** makes `Z ⟂ (Y(0), Y(1))`, so a plain difference in means is an **unbiased** ATE — you verified it over 400 experiments.
- **Power/MDE/sample size** trade off; you sized a test and confirmed the empirical power matched the target by simulation.
- **CUPED** (and covariate-adjusted OLS) cut the standard error using a pre-period covariate while leaving the estimate unbiased — free power.
- **Peeking** turns repeated looks into multiple testing and inflates the false-positive rate well above 5%.
- **ITT vs. per-protocol**: ITT is unbiased for the offer effect; per-protocol is confounded; `ITT / compliance` recovers the compliers' effect.

**You're ready for Week 4** if you can size a test, explain why CUPED is free power, and name the failure modes — peeking, SRM, and interference. Next week: causal graphs (DAGs) and what to adjust for when you *can't* randomize.